In [7]:
# Step 1: Import required libraries
import pandas as pd
import string

# Load your datasets
places = pd.read_csv("places.csv")
reviews = pd.read_csv("reviews.csv")

In [9]:
places.head()

,place_id,name,lat,lng,address,types,rating,user_ratings_total,phone,website,opening_hours_present,source_query,category
0,ChIJF2_wv4lHDW0R47WKR5aSdmE,Onslow,-36.847685,174.769957,"9 Princes Street, Auckland Central, Auckland 1...","establishment,food,point_of_interest,restaurant",4.8,1289.0,+64 9 930 9123,http://www.onslow.nz/,True,restaurant in Auckland,restaurant
1,ChIJKWPi4S5HDW0R-Ao9My5Mf48,Gilt Brasserie,-36.847655,174.767196,"2 Chancery Street, Chambers, Auckland 1010, Ne...","establishment,food,point_of_interest,restaurant",4.6,613.0,+64 9 300 3126,https://giltbrasserie.nz/,True,restaurant in Auckland,restaurant
2,ChIJjz8Dgf5HDW0RcKfgUZdV8fw,Amano,-36.844412,174.770464,"66 - 68, Tyler Street, Britomart Place, Auckla...","bakery,establishment,food,point_of_interest,re...",4.6,4358.0,+64 9 394 1416,https://savor.co.nz/amano,True,restaurant in Auckland,restaurant
3,ChIJdQU6o-RHDW0RBI93MBK5gcU,Hello Beasty,-36.843112,174.762463,"95-97 Customs Street West, Auckland Central, A...","establishment,food,point_of_interest,restaurant",4.7,1394.0,+64 21 554 496,http://hellobeasty.nz/,True,restaurant in Auckland,restaurant
4,ChIJKZLQLLRHDW0R_F9vtz1cjiQ,alma,-36.844086,174.769165,"130 Quay Street, Auckland Central, Auckland 10...","establishment,food,point_of_interest,restaurant",4.7,405.0,+64 9 242 1570,http://www.alma.nz/,True,restaurant in Auckland,restaurant


In [11]:
reviews.head()

,place_id,author_name,rating,text,time,relative_time,language,profile_photo_url,review_id
0,ChIJF2_wv4lHDW0R47WKR5aSdmE,Kiwi Kiwi,5,Fabulous Food!\nEnjoyed an outstanding food pe...,1754804453,in the last week,en,https://lh3.googleusercontent.com/a-/ALV-UjUZw...,NaN
1,ChIJY5ETuM2gEm0RFg6D9SQ6qsE,C P,5,Wonderful little store with an amazing variety...,1706345529,a year ago,en,NaN,385c974be3592a045667e1e1
2,ChIJY5ETuM2gEm0RFg6D9SQ6qsE,Matt owens,5,Bhanas is the type of store that makes you fee...,1504729714,7 years ago,en,NaN,0977732e5142d7a603182a63
3,ChIJY5ETuM2gEm0RFg6D9SQ6qsE,Tracey Hakaraia,5,Always such wonderful service from the Bhana F...,1639367210,3 years ago,en,NaN,dff371221b3a6d09c83382b5
4,ChIJY5ETuM2gEm0RFg6D9SQ6qsE,Linda Thornton,5,Best little grocery store in town. Lovely fami...,1661460203,2 years ago,en,NaN,074270790c7603b12f903e24


## Data Preprocessing

Group reviews

In [15]:
# Group reviews to summarize per place_id
reviews_grouped = (
    reviews
    .groupby("place_id", as_index=False)
    .agg({
        "review_id": "count",            # how many reviews per shop
        "text": lambda x: list(x)        # optional: keep list of review texts
    })
    .rename(columns={"review_id": "review_count"})
)
reviews_grouped.head()


,place_id,review_count,text
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi..."


In [17]:
# Merge grouped reviews with places
shops_r = reviews_grouped.merge(
    places[["place_id", "name", "address", "lat", "lng", "user_ratings_total"]],
    on="place_id",
    how="inner",
    validate="1:1"   # now one row per place
)
shops_r.head()


,place_id,review_count,text,name,address,lat,lng,user_ratings_total
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...,BurgerFuel Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821870,174.608392,1115.0
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi...",Westgate Takeaways,"579 Don Buck Road, Westgate, Massey North 0614...",-36.821860,174.607987,287.0


### Quick overview of the data

In [49]:
print("Places dataset-")
print(f"Rows: {places.shape[0]}, Columns: {places.shape[1]}")

print("Reviews dataset-")
print(f"Rows: {reviews.shape[0]}, Columns: {reviews.shape[1]}")

print("Grouped reviews dataset-")
print(f"Rows: {reviews_grouped.shape[0]}, Columns: {reviews_grouped.shape[1]}")

print("Merged dataset-")
print(f"Rows: {shops_r.shape[0]}, Columns: {shops_r.shape[1]}")

Places dataset-
Rows: 6501, Columns: 13
Reviews dataset-
Rows: 24805, Columns: 9
Grouped reviews dataset-
Rows: 5508, Columns: 3
Merged dataset-
Rows: 5508, Columns: 8


**Check for missingness**

In [52]:
# Count missing values in each column
shops_r.isna().sum()


place_id              0
review_count          0
text                  0
name                  0
address               0
lat                   0
lng                   0
user_ratings_total    2
dtype: int64

**Check for duplicates**

In [56]:
# Verify one unique row per place
duplicate_places = shops_r["place_id"].duplicated().sum()
print(f"Duplicate place_ids: {duplicate_places}")


Duplicate place_ids: 0


In [60]:
print("Max reviews seen:", shops_r["review_count"].max())
assert shops_r["review_count"].max() <= 5, "Unexpected: a place has >5 reviews"


Max reviews seen: 7


AssertionError: Unexpected: a place has >5 reviews

In [64]:
# Find places with >5 reviews
shops_r[shops_r["review_count"] > 5][["place_id", "name", "review_count"]]


,place_id,name,review_count
237,ChIJ1bK-dBhHDW0RE7wELDD6bPI,KFC,6
799,ChIJ7xnZXKdPDW0RFJEDMXgNN84,Sushi Time,6
1176,ChIJCVSo8DlNDW0RJHwR9Oef5m4,Hong Yuan Flat Bush,6
1842,ChIJJyUY-CtMDW0RUF8IBWA8vmU,KFC,7
2478,ChIJRXrk8StJDW0RqUz1eWIHIDU,Goode Brothers,6
2616,ChIJT5IiTwA5DW0RCSKENlno9ZY,Lazeez kebabs&burgers,6
2695,ChIJU4Wcf-VJDW0R6L38Gukj5dY,Kohi Fresh Fish & Takeaways,6
2723,ChIJUWwDwd4VDW0RYW_ydKl9u6U,Bombay Delights Indian & Fusion Cuisine,6
2805,ChIJVV8JMMhNDW0RWujcjSX0qt8,Burger King Papatoetoe,6
2831,ChIJVeL5rrBTDW0RRB50-nC9WA4,Papas Chicken,6


In [66]:
# Look at the actual review texts for one
pid = shops_r.loc[shops_r["review_count"] > 5, "place_id"].iloc[0]
for idx, review in enumerate(shops_r.loc[shops_r["place_id"] == pid, "text"].values[0], start=1):
    print(f"{idx}. {review}\n")


1. Terrible experience tonight drive through all round. The lady at the drive through was rushing us through our order. So much so, when we got to the window, she had split our order of two meals into two separate orders. When she gave us the bag of food, she shoved it so much the bag ripped and I had to cradle it to get it into the car without the food falling out. When we got home, the drinks (sprite) were flat and not cold, the zinger burger box wasn’t even in a box and the burger was almost coming out of the packaging. Half the meat was missing off my drumstick, and the meat remaining had no coating. The chips were soggy, but I’ll get them a star as they did put extra seasoning on. Our chicken, bun and burger was all luke warm so we had to put it in the microwave to heat it up. The only thing that was good was the potato and gravy. All in all, extremely disappointing and overpriced meal and will not be returning.

2. wanted to provide feedback about my recent visit to KFC Pt Chev. 

### Text Cleaning and Preprocessing

In [27]:
import re

def clean_review_text(text):
    """Normalize review text for NLP."""
    text = str(text).lower()                          # lowercase
    text = re.sub(r"https?://\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"@\w+", " ", text)                  # remove mentions
    text = re.sub(r"#\w+", " ", text)                  # remove hashtags
    text = re.sub(r"[^\w\s]", " ", text)               # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()           # normalize spaces
    return text

# Create a new column with cleaned reviews for each shop
shops_r["clean_texts"] = shops_r["text"].apply(lambda reviews: [clean_review_text(r) for r in reviews])
shops_r.head()


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,clean_texts
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,[we had a medium roast lamb and roast beef din...
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0,[excellent lovely chinese food plus they have ...
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0,"[safe secure easy launching, beautiful and tra..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...,BurgerFuel Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821870,174.608392,1115.0,[service was great guy at the cashier and serv...
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi...",Westgate Takeaways,"579 Don Buck Road, Westgate, Massey North 0614...",-36.821860,174.607987,287.0,[staff and place is great but noticed the chic...


In [23]:
shops_r.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...,BurgerFuel Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821870,174.608392,1115.0
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi...",Westgate Takeaways,"579 Don Buck Road, Westgate, Massey North 0614...",-36.821860,174.607987,287.0


### Tokenisation

In [39]:
import re

# A basic stopword list (can be expanded later)
STOPWORDS = set("""
a about above after again against all am an and any are as at be because been before being below
between both but by can did do does doing down during each few for from further had has have having
he her here hers herself him himself his how i if in into is it its itself just me more most my
myself no nor not of off on once only or other our ours ourselves out over own same she should so
some such than that the their theirs them themselves then there these they this those through to too
under until up very was we were what when where which while who whom why will with you your yours
yourself yourselves
""".split())

def tokenize_and_remove_stopwords(text):
    """Split text into tokens and remove common stopwords."""
    tokens = re.findall(r"[a-z']+", text)  # words only
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens


Clean empty reviews

In [37]:
# Remove shops where any review text in the list is empty or NaN
shops_r["clean_texts"] = shops_r["clean_texts"].apply(
    lambda reviews: [r for r in reviews if r.strip() not in ("", "nan")]
)

# Drop rows where the resulting list is empty (no valid reviews left)
shops_r = shops_r[shops_r["clean_texts"].apply(len) > 0].copy()


In [41]:
# Apply to each review in every shop
shops_r["tokens"] = shops_r["clean_texts"].apply(
    lambda reviews: [tokenize_and_remove_stopwords(r) for r in reviews]
)

shops_r.head(1)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,clean_texts,tokens
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi..."


In [43]:
# Look at tokens for a random shop
import random
sample_tokens = random.choice(shops_r["tokens"].values)
for i, review_tokens in enumerate(sample_tokens, start=1):
    print(f"Review {i}: {review_tokens}")


Review 1: ['small', 'park', 'next', 'car', 'park', 'swings', 'climbing', 'area', 'smaller', 'kids', 'chairs', 'benches', 'sit', 'shade', 'trees', 'grassed', 'area', 'flower', 'beds', 's', 'next', 'busy', 'road', 'car', 'park', 'need', 'careful', 'young', 'children']
Review 2: ['nicely', 'maintained', 'curated', 'small', 'park', 'middle', 'busy', 'takapuna', 'area', 'decent', 'sized', 'kids', 'play', 'area', 'along', 'white', 'benches', 'sit', 'relax', 'canopy', 'cover', 'pleasent', 'refreshing', 'public', 'toilets', 'also', 'available', 'park', 'location', 'easily', 'accessible', 'anywhere', 'takapuna']
Review 3: ['loved', 'taking', 'boy', 'quick', 'play', 'equipment', 'right', 'age', 'half', 'good', 'amount', 'green', 'space', 'normally', 'group', 'pigeons', 'gathered', 'great', 'fro', 'pigeon', 'feeding', 'car', 'park', 'pain', 'find', 's', 'walled', 'roads', 'take', 'care', 'taking', 'kids', 'otherwise', 'relaxing', 'nice', 'spot']
Review 4: ['nice', 'tidy', 'park', 'small', 'enough

Drop single character tokens

In [48]:
import re

def clean_tokens(tokens):
    out = []
    for t in tokens:
        if t == "s":               # drop possessive leftovers
            continue
        if len(t) < 2:             # drop 1-char tokens
            continue
        if re.fullmatch(r"\d+", t):# drop pure numbers
            continue
        out.append(t)
    return out

# apply to each review’s token list
shops_r["tokens_clean"] = shops_r["tokens"].apply(lambda reviews: [clean_tokens(toks) for toks in reviews])
shops_r = shops_r.drop('tokens', axis=1)
shops_r.head(1)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,clean_texts,tokens_clean
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi..."


**Lemmatization (Normalize)**

In [51]:
# Requires NLTK once; run these two lines only the first time
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

shops_r["tokens_lemma"] = shops_r["tokens_clean"].apply(
    lambda reviews: [lemmatize_tokens(toks) for toks in reviews]
)
shops_r = shops_r.drop('tokens_clean', axis=1)
shops_r.head(1)


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Shubham\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Shubham\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,clean_texts,tokens_lemma
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi..."


## Sentiment analysis

In [54]:
!pip install vaderSentiment


In [56]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# initialize analyzer
analyzer = SentimentIntensityAnalyzer()


In [58]:
def score_review(text):
    """Return compound score from VADER (-1 to +1)."""
    return analyzer.polarity_scores(text)["compound"]

# Apply to each shop: one list of review scores per place
shops_r["sentiment_scores"] = shops_r["clean_texts"].apply(
    lambda reviews: [score_review(r) for r in reviews]
)

shops_r.head(1)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,clean_texts,tokens_lemma,sentiment_scores
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi...","[0.9432, 0.3891, 0.6361, 0.5849, 0.5719]"


In [60]:
import numpy as np

def aggregate_sentiment(scores):
    """Return average, min, max, and percentage of positive reviews."""
    avg = np.mean(scores) if scores else 0
    pos_pct = sum(s >= 0.05 for s in scores) / len(scores) * 100 if scores else 0
    neg_pct = sum(s <= -0.05 for s in scores) / len(scores) * 100 if scores else 0
    return avg, pos_pct, neg_pct

shops_r[["avg_sentiment", "pct_positive", "pct_negative"]] = shops_r["sentiment_scores"].apply(
    lambda scores: pd.Series(aggregate_sentiment(scores))
)

shops_r.head(3)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,clean_texts,tokens_lemma,sentiment_scores,avg_sentiment,pct_positive,pct_negative
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi...","[0.9432, 0.3891, 0.6361, 0.5849, 0.5719]",0.62504,100.0,0.0
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0,[excellent lovely chinese food plus they have ...,"[[excellent, lovely, chinese, food, plus, grea...","[0.9493, 0.9169, 0.7447, 0.886, -0.296]",0.64018,80.0,20.0
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0,"[safe secure easy launching, beautiful and tra...","[[safe, secure, easy, launching], [beautiful, ...","[0.802, 0.6249, 0.4215, 0.5994, 0.4456]",0.57868,100.0,0.0


In [64]:
for idx, review in enumerate(shops_r.loc[shops_r["place_id"] == "ChIJ-00UHBxKDW0RT8IBRwA5KsM", "tokens_lemma"].values[0], start=1):
    print(f"{idx}. {review}\n")



1. ['excellent', 'lovely', 'chinese', 'food', 'plus', 'great', 'option', 'thai', 'also', 'portion', 'great']

2. ['fish', 'chip', 'yummy', 'fish', 'fresh', 'crispy', 'batter', 'big', 'piece', 'chip', 'nicely', 'cooked', 'soft', 'inside', 'crispy', 'outside', 'reasonable', 'price', 'good', 'value', 'money']

3. ['small', 'portion', 'expensive', 'get', 'also', 'got', 'stir', 'fried', 'chicken', 'dish', 'tasted', 'funky', 'like', 'seafood', 'bite', 'didn', 'want', 'keep', 'eating', 'tried', 'giving', 'feedback', 'phone', 'said', 'never', 'complaint']

4. ['st', 'heliers', 'takeaway', 'maskell', 'st', 'awesome', 'best', 'thai', 'chinese', 'european', 'takeaway', 'auckland', 'gai', 'sam', 'ro', 'dish', 'exquisite', 'fact', 'yet', 'meal', 'hasn', 'want', 'top', 'quality', 'food', 'decent', 'price', 'need', 'come']

5. ['order', 'beef', 'pad', 'thai', 'got', 'something', 'completely', 'different', 'shrimp', 'fat', 'beef', 'extra', 'pepper', 'mind', 'allergic', 'pepper']



**TUNE VADER WITH DOMAIN BASED LEXICON**

In [67]:
# A2) Extend VADER lexicon with domain-specific weights (negatives stronger)
domain_updates = {
    "expensive": -2.2,
    "overpriced": -2.6,
    "pricey": -1.8,
    "small": -0.7,          # smaller portions in food context
    "portion": -0.5,
    "portions": -0.7,
    "funky": -2.4,          # off-taste / smell
    "stale": -2.6,
    "complaint": -2.2,
    "complaints": -2.4,
    "inedible": -3.2,
    "cold": -1.6,           # cold food (contextual; adjust if false positives)
    "bland": -2.0,
    "greasy": -1.6,
}
analyzer.lexicon.update(domain_updates)


In [72]:
# A3) Helper to score one review with tuned VADER
def vader_score(text: str) -> float:
    return analyzer.polarity_scores(str(text))["compound"]

# quick test on your review
sample_text = """['small', 'portion', 'expensive', 'get', 'also', 'got', 'stir', 'fried', 'chicken', 'dish', 'tasted', 'funky', 'like', 'seafood',
        'bite', 'didn', 'want', 'keep', 'eating', 'tried', 'giving', 'feedback', 'phone', 'said', 'never', 'complaint']"""
vader_score(sample_text)


-0.2434